In [1]:
%%capture
!pip install timm
!pip install fastai

In [2]:
from fastai.vision.all import * #import everthing from vision
import pandas as pd
import timm
import time

In [3]:
start_time = time.time()

In [4]:
def random_seed(seed_value, use_cuda):
    np.random.seed(seed_value)
 #cpu vars
    torch.manual_seed(seed_value)
# cpu  vars
    random.seed(seed_value)
 # Python
    if use_cuda:
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value)
# gpu vars
        torch.backends.cudnn.deterministic = True
 #needed
        torch.backends.cudnn.benchmark = False
#Remember to use num_workers=0 when creating the DataBunch.

In [5]:
# You can use almost any integer (less than the number of random generators on your pc)
random_seed(2026,True) #Let's use your fav year

In [6]:
train_df = pd.read_csv("/kaggle/input/datasets/victorolufemi/spot-the-mask-challenge/train_labels.csv")
submission = pd.read_csv("/kaggle/input/datasets/victorolufemi/spot-the-mask-challenge/SampleSubmission.csv")
Nosemask = DataBlock(blocks=(ImageBlock, CategoryBlock), splitter=TrainTestSplitter(0.1, stratify=train_df["target"]),get_x = ColReader(0, pref = "/kaggle/input/datasets/victorolufemi/spot-the-mask-challenge/images/images/" ),get_y=ColReader(1),item_tfms = Resize(460), batch_tfms = aug_transforms(do_flip=True,flip_vert=True,max_lighting=0.4,max_zoom=1.2,max_warp=0.2,max_rotate=30,xtra_tfms=None))
dls = Nosemask.dataloaders(train_df, bs=16, num_workers=0)
learn = vision_learner(dls,'convnext_tiny', metrics=[accuracy], path=".") #try convnext_base and convnext_large

learn.remove_cb(ProgressCallback) # remove notebook progress bar (causes crash in Kaggle)

# Use both GPUs with DataParallel
learn.model = torch.nn.DataParallel(learn.model)

learn.fine_tune(3,cbs=MixUp) #Apply Mixup #increase no of Epochs
tdl = dls.test_dl(submission) # quicly create test data loader
test_preds_, test_labels_ = learn.get_preds(dl=tdl) #Get Preds
submission['target'] = [float(p[1]) for p in test_preds_]
submission.to_csv('convnext_tiny_model.csv', index=False)

/usr/local/lib/python3.12/dist-packages/fastai/data/transforms.py:214: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  o = r[c] if isinstance(c, int) or not c in getattr(r, '_fields', []) else getattr(r, c)
/usr/local/lib/python3.12/dist-packages/fastai/data/transforms.py:214: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  o = r[c] if isinstance(c, int) or not c in getattr(r, '_fields', []) else getattr(r, c)


model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/fastai/data/transforms.py:214: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  o = r[c] if isinstance(c, int) or not c in getattr(r, '_fields', []) else getattr(r, c)


[0, 0.905406653881073, 0.09493153542280197, 0.9770992398262024, '03:24']
[0, 0.6476715207099915, 0.049508120864629745, 0.9847328066825867, '04:00']
[1, 0.5554237365722656, 0.024380188435316086, 0.9923664331436157, '04:00']
[2, 0.5271216630935669, 0.03661230579018593, 0.9923664331436157, '03:59']


In [7]:
end_time = time.time()
elapsed = end_time - start_time

hours, rem = divmod(elapsed, 3600)
minutes, seconds = divmod(rem, 60)
print(f"Total runtime: {int(hours):02d}h {int(minutes):02d}m {int(seconds):02d}s")

Total runtime: 00h 16m 41s
